# HW14 — Lifelong Learning
**NTU ML 2021 Spring**

目標：訓練一個模型依序學習 5 個 Task，且學完新 Task 後不遺忘舊 Task（避免 Catastrophic Forgetting）。

## 方法總覽
| 方法 | 類別 | 核心概念 |
|------|------|---------|
| Baseline | — | 直接依序訓練，不做防遺忘 |
| EWC | Regularization | 用 Fisher Information 保護重要參數 |
| MAS | Regularization | 用 output L2-norm 的梯度衡量參數重要性 |
| SI | Regularization | 累積 online 路徑長度估計重要性 |
| RWalk | Regularization | EWC + SI 的混合 |
| SCP | Regularization | Sliced Cramer Preservation |


In [ ]:
# 確保 tqdm 版本足夠新
# !pip install -q tqdm torchvision

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import tqdm


## Utility

利用 Permuted MNIST 建立 5 個不同 Task。

In [ ]:
# ──────────────────────────────────────────────
# Permutation：對圖片像素位置做隨機重排
# ──────────────────────────────────────────────
def _permutate_image_pixels(image, permutation):
    """依照 permutation 重排圖片像素（channel 軸不動）"""
    if permutation is None:
        return image
    c, h, w = image.size()
    image = image.view(-1, c)       # (H*W, C)
    image = image[permutation, :]   # 重排 pixel 順序
    image = image.view(c, h, w)    # 還原 (C, H, W) ← 原版少了這行賦值，已修正
    return image

def get_transform(permutation=None, normalize=True):
    ops = [transforms.ToTensor(), Pad(28)]
    if normalize:
        ops.append(transforms.Normalize((0.1307,), (0.3081,)))
    ops.append(transforms.Lambda(lambda x: _permutate_image_pixels(x, permutation)))
    return transforms.Compose(ops)

class Pad:
    """將圖片 zero-pad 到指定大小（MNIST 已是 28×28，此處為 no-op）"""
    def __init__(self, size, fill=0, padding_mode='constant'):
        self.size = size
        self.fill = fill
        self.padding_mode = padding_mode

    def __call__(self, img):
        img_size = img.size()[1]
        assert (self.size - img_size) % 2 == 0, "padding 必須對稱"
        pad = (self.size - img_size) // 2
        return F.pad(img, (pad, pad, pad, pad), self.padding_mode, self.fill)

class Data:
    """封裝 MNIST dataset，支援 pixel permutation"""
    def __init__(self, path, train=True, permutation=None, normalize=True):
        self.dataset = datasets.MNIST(
            root=os.path.join(path, 'MNIST'),
            transform=get_transform(permutation, normalize),
            train=train,
            download=True,
        )


In [ ]:
# ──────────────────────────────────────────────
# 超參數設定
# ──────────────────────────────────────────────
class Args:
    task_number     = 5      # Task 數量
    epochs_per_task = 10     # 每個 Task 訓練幾個 epoch
    lr              = 1e-4   # Adam Learning Rate
    batch_size      = 128
    test_size       = 8192   # 每次 evaluate 取多少測試樣本
    random_seed     = 0

args = Args()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用裝置：{device}')

# ──────────────────────────────────────────────
# 產生 5 種 permutation（Task 0 保持原始 MNIST）
# ──────────────────────────────────────────────
np.random.seed(args.random_seed)
permutations = [
    np.random.permutation(784) if idx != 0 else np.arange(784)
    for idx in range(args.task_number)
]

# ──────────────────────────────────────────────
# 建立 DataLoader
# ──────────────────────────────────────────────
train_datasets = [Data('data', permutation=p)               for p in permutations]
test_datasets  = [Data('data', train=False, permutation=p)  for p in permutations]

train_dataloaders = [DataLoader(d.dataset, batch_size=args.batch_size, shuffle=True)
                     for d in train_datasets]
test_dataloaders  = [DataLoader(d.dataset, batch_size=args.test_size, shuffle=False)
                     for d in test_datasets]

print(f'共 {args.task_number} 個 Task，每 Task 訓練 {args.epochs_per_task} epoch')


### Model

固定模型架構：`784 → 1024 → 512 → 256 → 10`（全部 Task 共用同一個 model）

In [ ]:
class Model(nn.Module):
    """固定大小的 4 層全連接網路"""
    def __init__(self):
        super().__init__()
        self.fc1  = nn.Linear(784,  1024)
        self.fc2  = nn.Linear(1024, 512)
        self.fc3  = nn.Linear(512,  256)
        self.fc4  = nn.Linear(256,  10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = x.view(-1, 784)           # 攤平圖片為向量
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        return self.fc4(x)             # 最後一層不接 activation（CrossEntropy 會處理）

print(Model())


### Train

通用訓練函式。所有 Regularization-based 方法都用同一個 `train()`，差異在 `lll_object` 計算不同的懲罰項。

In [ ]:
def train(model, optimizer, dataloader, epochs_per_task,
          lll_object, lll_lambda, test_dataloaders, evaluate, device):
    """
    通用訓練迴圈。
    total_loss = CrossEntropyLoss + lll_lambda * lll_object.penalty(model)
    """
    model.train()
    model.zero_grad()
    criterion     = nn.CrossEntropyLoss()
    acc_per_epoch = []

    bar = tqdm.auto.trange(epochs_per_task, leave=False, desc='Epoch 1')
    for epoch in bar:
        for imgs, labels in tqdm.auto.tqdm(dataloader, leave=False):
            imgs, labels = imgs.to(device), labels.to(device)

            outputs    = model(imgs)
            task_loss  = criterion(outputs, labels)
            # lll_object.penalty：Lifelong Learning 正規化懲罰項（防止遺忘）
            total_loss = task_loss + lll_lambda * lll_object.penalty(model)

            # 部分方法（SI / RWalk）需在 backward 前先 update 內部累積梯度統計量
            lll_object.update(model)

            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()

            bar.set_description_str(
                f'Epoch {epoch+1:2}, Loss: {total_loss.item():.5f}', refresh=True)

        # epoch 結束，計算所有已學 Task 的平均準確率
        accs = [evaluate(model, dl, device) for dl in test_dataloaders]
        acc_per_epoch.append(np.mean(accs) * 100.0)

    return model, optimizer, acc_per_epoch


### Evaluate

計算模型在指定 dataloader 上的分類準確率。

In [ ]:
def evaluate(model, test_dataloader, device):
    """回傳 0~1 之間的準確率"""
    model.eval()
    correct = 0
    total   = 0
    with torch.no_grad():
        for imgs, labels in test_dataloader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs      = model(imgs)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total   += labels.numel()
    return correct / total


## Visualization

視覺化前 3 個 Task 的 Permuted MNIST 樣本（每個 label 各取 1 張）。

In [ ]:
sample = [Data('data', permutation=permutations[i], normalize=False) for i in range(3)]

plt.figure(figsize=(22, 7))
for task in range(3):
    # targets 在新版 torchvision 是 Tensor，需用 .tolist() 轉成 Python int list
    targets = sample[task].dataset.targets.tolist()
    indices = [targets.index(l) for l in range(10)]

    for col, idx in enumerate(indices):
        ax = plt.subplot(3, 10, task * 10 + col + 1)
        img = sample[task].dataset[idx][0].squeeze().numpy()
        plt.imshow(img, cmap='gray')
        plt.axis('off')
        if task == 0:
            ax.set_title(f'label {col}', fontsize=8)
        if col == 0:
            ax.set_ylabel(f'Task {task+1}', fontsize=9, rotation=0, labelpad=35)

plt.suptitle('Permuted MNIST — 前 3 個 Task 各 label 樣本', fontsize=13)
plt.tight_layout()
plt.show()


## Methods

每個方法都繼承相同的介面，實作三個函式：
- `_calculate_importance()` / `calculate_importance()`：計算參數重要性矩陣
- `penalty(model)`：計算正規化懲罰項，加入 total_loss
- `update(model)`：每個 batch 後更新內部狀態（Baseline / EWC / MAS / SCP 為 no-op）


### Baseline

不做任何防遺忘處理，所有重要性權重都是 0。用於對照。

In [ ]:
class baseline:
    """
    不加任何正規化懲罰（precision_matrices 全為 0），
    直接依序訓練 → 嚴重 Catastrophic Forgetting。
    """
    def __init__(self, model, dataloaders, device):
        self.model   = model
        self.device  = device
        self.params  = {n: p for n, p in model.named_parameters() if p.requires_grad}
        self.p_old   = {}
        self._precision_matrices = self._calculate_importance()

        for n, p in self.params.items():
            self.p_old[n] = p.clone().detach()

    def _calculate_importance(self):
        # 全部填 0 → penalty 永遠為 0
        return {n: p.clone().detach().fill_(0) for n, p in self.params.items()}

    def penalty(self, model):
        loss = 0
        for n, p in model.named_parameters():
            loss += (self._precision_matrices[n] * (p - self.p_old[n]) ** 2).sum()
        return loss

    def update(self, model):
        return  # do nothing


#### Baseline — 執行訓練

In [ ]:
print('RUN BASELINE')
model     = Model().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)

lll_object  = baseline(model=model, dataloaders=[None], device=device)
lll_lambda  = 0.0
baseline_acc = []
task_bar = tqdm.auto.trange(args.task_number, desc='Task 1')

for i in task_bar:
    model, _, acc_list = train(
        model, optimizer, train_dataloaders[i], args.epochs_per_task,
        lll_object, lll_lambda, test_dataloaders[:i+1], evaluate, device)

    lll_object = baseline(model=model, dataloaders=test_dataloaders[:i], device=device)
    optimizer  = torch.optim.Adam(model.parameters(), lr=args.lr)
    baseline_acc.extend(acc_list)
    task_bar.set_description_str(f'Task {i+2}')

print(f'Baseline 最終 Average Accuracy: {baseline_acc[-1]:.2f}%')
print('=' * 80)


### EWC — Elastic Weight Consolidation

Loss 函式：

$$\mathcal{L}_B = \mathcal{L}(\theta) + \sum_i \frac{\lambda}{2} F_i (\theta_i - \theta_{A,i}^*)^2$$

$F_i$ 是 Fisher Information Matrix 的對角線元素，代表第 $i$ 個參數對舊 Task 的重要性。

$$F = \mathbb{E}\left[\left(\nabla \log p(y|x, \theta_A^*)\right)^2\right]$$

只取對角線近似（即每次 backward 取 $\nabla \log p$ 的平方再平均）。


In [ ]:
class ewc:
    """
    Elastic Weight Consolidation (Kirkpatrick et al., 2017)
    用 Fisher Information 衡量參數對舊 Task 的重要性，懲罰重要參數的改變。
    """
    def __init__(self, model, dataloaders, device):
        self.model       = model
        self.dataloaders = dataloaders
        self.device      = device
        self.params      = {n: p for n, p in model.named_parameters() if p.requires_grad}
        self.p_old       = {}
        self._precision_matrices = self._calculate_importance()

        for n, p in self.params.items():
            self.p_old[n] = p.clone().detach()

    def _calculate_importance(self):
        """計算 Fisher Information Matrix（對角線近似）"""
        precision_matrices = {n: p.clone().detach().fill_(0) for n, p in self.params.items()}

        self.model.eval()
        if self.dataloaders[0] is not None:
            num_data = sum(len(dl) for dl in self.dataloaders)
            for dataloader in self.dataloaders:
                for imgs, labels in dataloader:
                    self.model.zero_grad()
                    imgs, labels = imgs.to(self.device), labels.to(self.device)
                    output = self.model(imgs)
                    # 用 NLL loss 計算 log-likelihood，backward 取梯度
                    loss = F.nll_loss(F.log_softmax(output, dim=1), labels)
                    loss.backward()
                    for n, p in self.model.named_parameters():
                        # Fisher 對角線 = 梯度平方的期望值
                        precision_matrices[n].data += p.grad.data ** 2 / num_data

        return precision_matrices

    def penalty(self, model):
        loss = 0
        for n, p in model.named_parameters():
            loss += (self._precision_matrices[n] * (p - self.p_old[n]) ** 2).sum()
        return loss

    def update(self, model):
        return  # EWC 不需要 per-batch 更新


#### EWC — 執行訓練

In [ ]:
print('RUN EWC')
model     = Model().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)

lll_object = ewc(model=model, dataloaders=[None], device=device)
lll_lambda = 100
ewc_acc    = []
task_bar   = tqdm.auto.trange(args.task_number, desc='Task 1')

for i in task_bar:
    model, _, acc_list = train(
        model, optimizer, train_dataloaders[i], args.epochs_per_task,
        lll_object, lll_lambda, test_dataloaders[:i+1], evaluate, device)

    # 用「所有學過的 task」重新算 Fisher Matrix
    lll_object = ewc(model=model, dataloaders=test_dataloaders[:i+1], device=device)
    optimizer  = torch.optim.Adam(model.parameters(), lr=args.lr)
    ewc_acc.extend(acc_list)
    task_bar.set_description_str(f'Task {i+2}')

print(f'EWC 最終 Average Accuracy: {ewc_acc[-1]:.2f}%')
print('=' * 80)


### MAS — Memory Aware Synapses

Loss 函式與 EWC 相同，差異在 $\Omega_i$ 的計算方式：

$$\Omega_i = \left\| \frac{\partial \|M(x; \theta)\|^2}{\partial \theta_i} \right\|$$

不需要 label，用 **output 向量的 L2-norm 平方** 的梯度衡量參數重要性。


In [ ]:
class mas:
    """
    Memory Aware Synapses (Aljundi et al., 2018)
    不需要 label，透過 output L2-norm 的梯度衡量參數重要性。
    """
    def __init__(self, model, dataloaders, device):
        self.model       = model
        self.dataloaders = dataloaders
        self.device      = device
        self.params      = {n: p for n, p in model.named_parameters() if p.requires_grad}
        self.p_old       = {}
        self._precision_matrices = self._calculate_importance()

        for n, p in self.params.items():
            self.p_old[n] = p.clone().detach()

    def _calculate_importance(self):
        """計算 Omega 矩陣：output L2-norm 平方對參數的梯度絕對值"""
        precision_matrices = {n: p.clone().detach().fill_(0) for n, p in self.params.items()}

        self.model.eval()
        if self.dataloaders[0] is not None:
            num_data = sum(len(dl) for dl in self.dataloaders)
            for dataloader in self.dataloaders:
                for imgs, _ in dataloader:          # 不需要 label
                    self.model.zero_grad()
                    output = self.model(imgs.to(self.device))

                    # output 每個元素平方，再對整個 batch 求和後取平均
                    loss = output.pow(2).sum(dim=1).mean()
                    loss.backward()

                    for n, p in self.model.named_parameters():
                        # 取梯度的絕對值（與 EWC 取平方不同）
                        precision_matrices[n].data += p.grad.abs() / num_data

        return precision_matrices

    def penalty(self, model):
        loss = 0
        for n, p in model.named_parameters():
            loss += (self._precision_matrices[n] * (p - self.p_old[n]) ** 2).sum()
        return loss

    def update(self, model):
        return


#### MAS — 執行訓練

In [ ]:
print('RUN MAS')
model     = Model().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)

lll_object = mas(model=model, dataloaders=[None], device=device)
lll_lambda = 0.1
mas_acc    = []
task_bar   = tqdm.auto.trange(args.task_number, desc='Task 1')

for i in task_bar:
    model, _, acc_list = train(
        model, optimizer, train_dataloaders[i], args.epochs_per_task,
        lll_object, lll_lambda, test_dataloaders[:i+1], evaluate, device)

    lll_object = mas(model=model, dataloaders=test_dataloaders[:i+1], device=device)
    optimizer  = torch.optim.Adam(model.parameters(), lr=args.lr)
    mas_acc.extend(acc_list)
    task_bar.set_description_str(f'Task {i+2}')

print(f'MAS 最終 Average Accuracy: {mas_acc[-1]:.2f}%')
print('=' * 80)


### SI — Synaptic Intelligence

在訓練**過程中**（online）累積每個參數對 loss 下降的貢獻度 $W$，
Task 結束後用 $W$ 除以參數變化量的平方估計重要性 $\Omega$。

不需要額外資料，完全 online 計算。


In [ ]:
class si:
    """
    Synaptic Intelligence (Zenke et al., 2017)
    Online 累積參數對 loss 下降的貢獻，估計每個 synapse 的重要性。
    """
    def __init__(self, model, dataloaders, epsilon, device):
        self.model       = model
        self.dataloaders = dataloaders
        self.device      = device
        self.epsilon     = epsilon   # 防止除以 0 的小常數
        self.params      = {n: p for n, p in model.named_parameters() if p.requires_grad}

        # _n_p_prev: 上一個 task 結束時的參數值
        # _n_omega:  累積的重要性權重
        self._n_p_prev, self._n_omega = self._calculate_importance()
        # W: 當前 task 內累積的「路徑貢獻」
        # p_old: 上一個 batch 的參數值（用來算 Δθ）
        self.W, self.p_old = self._init_W()

    def _init_W(self):
        W     = {}
        p_old = {}
        for n, p in self.model.named_parameters():
            key = n.replace('.', '__')
            if p.requires_grad:
                W[key]     = p.data.clone().zero_()
                p_old[key] = p.data.clone()
        return W, p_old

    def _calculate_importance(self):
        n_p_prev = {}
        n_omega  = {}

        if self.dataloaders[0] is not None:
            for n, p in self.model.named_parameters():
                key = n.replace('.', '__')
                if p.requires_grad:
                    p_prev    = getattr(self.model, f'{key}_SI_prev_task')
                    W         = getattr(self.model, f'{key}_W')
                    p_current = p.detach().clone()
                    p_change  = p_current - p_prev
                    omega_add = W / (p_change ** 2 + self.epsilon)
                    try:
                        omega = getattr(self.model, f'{key}_SI_omega')
                    except AttributeError:
                        omega = p.detach().clone().zero_()
                    omega_new     = omega + omega_add
                    n_omega[key]  = omega_new
                    n_p_prev[key] = p_current
                    self.model.register_buffer(f'{key}_SI_prev_task', p_current)
                    self.model.register_buffer(f'{key}_SI_omega',     omega_new)
        else:
            # 第一個 task：初始化 buffer，omega 全為 0
            for n, p in self.model.named_parameters():
                key = n.replace('.', '__')
                if p.requires_grad:
                    n_p_prev[key] = p.detach().clone()
                    n_omega[key]  = p.detach().clone().zero_()
                    self.model.register_buffer(f'{key}_SI_prev_task', p.detach().clone())

        return n_p_prev, n_omega

    def penalty(self, model):
        loss = 0.0
        for n, p in model.named_parameters():
            key = n.replace('.', '__')
            if p.requires_grad:
                loss += (self._n_omega[key] * (p - self._n_p_prev[key]) ** 2).sum()
        return loss

    def update(self, model):
        """每個 batch 後更新 W（累積梯度 × 參數位移）"""
        for n, p in model.named_parameters():
            key = n.replace('.', '__')
            if p.requires_grad:
                if p.grad is not None:
                    # W += -grad * (θ_current - θ_prev_batch)
                    self.W[key].add_(-p.grad * (p.detach() - self.p_old[key]))
                    self.model.register_buffer(f'{key}_W', self.W[key])
                self.p_old[key] = p.detach().clone()


#### SI — 執行訓練

In [ ]:
print('RUN SI')
model     = Model().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)

lll_object = si(model=model, dataloaders=[None], epsilon=0.1, device=device)
lll_lambda = 1
si_acc     = []
task_bar   = tqdm.auto.trange(args.task_number, desc='Task 1')

for i in task_bar:
    model, _, acc_list = train(
        model, optimizer, train_dataloaders[i], args.epochs_per_task,
        lll_object, lll_lambda, test_dataloaders[:i+1], evaluate, device)

    lll_object = si(model=model, dataloaders=test_dataloaders[:i+1], epsilon=0.1, device=device)
    optimizer  = torch.optim.Adam(model.parameters(), lr=args.lr)
    si_acc.extend(acc_list)
    task_bar.set_description_str(f'Task {i+2}')

print(f'SI 最終 Average Accuracy: {si_acc[-1]:.2f}%')
print('=' * 80)


### RWalk — Riemannian Walk

結合 EWC（Fisher Information）與 SI（online 路徑貢獻）的混合方法：

$$\Omega_i^{\text{RWalk}} = 0.5 \cdot \Omega_i^{\text{old}} + 0.5 \cdot \frac{W_i}{\frac{1}{2} F_i (\Delta\theta_i)^2 + \epsilon}$$

penalty 同時考慮 Fisher 矩陣和累積 omega：

$$\text{penalty} = \sum_i (\Omega_i + F_i)(\theta_i - \theta_{A,i}^*)^2$$


In [ ]:
class rwalk:
    """
    Riemannian Walk (Chaudhry et al., 2018)
    EWC（Fisher Matrix）+ SI（online 路徑貢獻）的混合。
    """
    def __init__(self, model, dataloaders, epsilon, device):
        self.model               = model
        self.dataloaders         = dataloaders
        self.device              = device
        self.epsilon             = epsilon
        self.update_ewc_param    = 0.4   # Fisher Matrix 的 EMA 更新係數
        self.params              = {n: p for n, p in model.named_parameters() if p.requires_grad}

        # 先計算 EWC Fisher Matrix
        self._precision_matrices = self._calculate_importance_ewc()
        # 再計算 SI omega
        self._n_p_prev, self._n_omega = self._calculate_importance()
        self.W, self.p_old = self._init_W()

    def _init_W(self):
        W, p_old = {}, {}
        for n, p in self.model.named_parameters():
            key = n.replace('.', '__')
            if p.requires_grad:
                W[key]     = p.data.clone().zero_()
                p_old[key] = p.data.clone()
        return W, p_old

    def _calculate_importance_ewc(self):
        """計算 Fisher Information Matrix（帶 EMA 更新）"""
        precision_matrices = {}
        for n, p in self.params.items():
            key = n.replace('.', '__')
            precision_matrices[key] = p.clone().detach().fill_(0)

        self.model.eval()
        if self.dataloaders[0] is not None:
            num_data = sum(len(dl) for dl in self.dataloaders)
            for dataloader in self.dataloaders:
                # EMA decay：先將舊的 Fisher 乘以 (1 - alpha)
                for n, p in self.model.named_parameters():
                    key = n.replace('.', '__')
                    precision_matrices[key].data *= (1 - self.update_ewc_param)
                for imgs, labels in dataloader:
                    self.model.zero_grad()
                    imgs, labels = imgs.to(self.device), labels.to(self.device)
                    output = self.model(imgs)
                    loss   = F.nll_loss(F.log_softmax(output, dim=1), labels)
                    loss.backward()
                    for n, p in self.model.named_parameters():
                        key = n.replace('.', '__')
                        precision_matrices[key].data += (
                            self.update_ewc_param * p.grad.data ** 2 / num_data)

        return precision_matrices

    def _calculate_importance(self):
        """計算 SI-style omega（整合 Fisher Matrix 的 RWalk 版本）"""
        n_p_prev, n_omega = {}, {}

        if self.dataloaders[0] is not None:
            for n, p in self.model.named_parameters():
                key = n.replace('.', '__')
                if p.requires_grad:
                    p_prev    = getattr(self.model, f'{key}_SI_prev_task')
                    W         = getattr(self.model, f'{key}_W')
                    p_current = p.detach().clone()
                    p_change  = p_current - p_prev
                    # 分母加入 Fisher Matrix 的影響（與純 SI 不同）
                    denom     = 0.5 * self._precision_matrices[key] * p_change ** 2 + self.epsilon
                    omega_add = W / denom
                    try:
                        omega = getattr(self.model, f'{key}_SI_omega')
                    except AttributeError:
                        omega = p.detach().clone().zero_()
                    # EMA 更新 omega
                    omega_new     = 0.5 * omega + 0.5 * omega_add
                    n_omega[key]  = omega_new
                    n_p_prev[key] = p_current
                    self.model.register_buffer(f'{key}_SI_prev_task', p_current)
                    self.model.register_buffer(f'{key}_SI_omega',     omega_new)
        else:
            for n, p in self.model.named_parameters():
                key = n.replace('.', '__')
                if p.requires_grad:
                    n_p_prev[key] = p.detach().clone()
                    n_omega[key]  = p.detach().clone().zero_()
                    self.model.register_buffer(f'{key}_SI_prev_task', p.detach().clone())

        return n_p_prev, n_omega

    def penalty(self, model):
        loss = 0.0
        for n, p in model.named_parameters():
            key = n.replace('.', '__')
            if p.requires_grad:
                # 同時考慮 omega 和 Fisher Matrix
                loss += ((self._n_omega[key] + self._precision_matrices[key])
                         * (p - self._n_p_prev[key]) ** 2).sum()
        return loss

    def update(self, model):
        """每個 batch 後更新 W"""
        for n, p in model.named_parameters():
            key = n.replace('.', '__')
            if p.requires_grad:
                if p.grad is not None:
                    self.W[key].add_(-p.grad * (p.detach() - self.p_old[key]))
                    self.model.register_buffer(f'{key}_W', self.W[key])
                self.p_old[key] = p.detach().clone()


#### RWalk — 執行訓練

In [ ]:
print('RUN RWalk')
model     = Model().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)

lll_object = rwalk(model=model, dataloaders=[None], epsilon=0.1, device=device)
lll_lambda = 100
rwalk_acc  = []
task_bar   = tqdm.auto.trange(args.task_number, desc='Task 1')

for i in task_bar:
    model, _, acc_list = train(
        model, optimizer, train_dataloaders[i], args.epochs_per_task,
        lll_object, lll_lambda, test_dataloaders[:i+1], evaluate, device)

    lll_object = rwalk(model=model, dataloaders=test_dataloaders[:i+1], epsilon=0.1, device=device)
    optimizer  = torch.optim.Adam(model.parameters(), lr=args.lr)
    rwalk_acc.extend(acc_list)
    task_bar.set_description_str(f'Task {i+2}')

print(f'RWalk 最終 Average Accuracy: {rwalk_acc[-1]:.2f}%')
print('=' * 80)


### SCP — Sliced Cramer Preservation

用「Sliced Cramer Distance」衡量參數重要性：
1. 對 batch 的 output 取平均向量 $\varphi$
2. 隨機抽取 $L$ 個單位球面向量 $\xi$
3. 對每個 $\xi$，計算內積 $\rho = \xi^T \varphi$，backward 取梯度
4. 各參數梯度的 $L$ 次平均絕對值 = $\Gamma$ 矩陣


In [ ]:
def sample_spherical(npoints, ndim):
    """在 ndim 維單位球面上隨機採樣 npoints 個向量"""
    vec = np.random.randn(ndim, npoints)
    vec /= np.linalg.norm(vec, axis=0, keepdims=True)
    return torch.from_numpy(vec)   # shape: (ndim, npoints)

class scp:
    """
    Sliced Cramer Preservation (SCP)
    不需要 label，透過 Sliced Cramer Distance 的梯度衡量參數重要性。
    """
    def __init__(self, model, dataloaders, L, device):
        self.model        = model
        self.dataloaders  = dataloaders
        self.device       = device
        self.L            = L          # 隨機方向採樣數量
        self.params       = {n: p for n, p in model.named_parameters() if p.requires_grad}
        self._state_params = {}
        self._precision_matrices = self._calculate_importance()

        for n, p in self.params.items():
            self._state_params[n] = p.clone().detach()

    def _calculate_importance(self):
        """計算 Gamma（Γ）矩陣"""
        precision_matrices = {n: p.clone().detach().fill_(0) for n, p in self.params.items()}

        self.model.eval()
        if self.dataloaders[0] is not None:
            num_data = sum(len(dl) for dl in self.dataloaders)
            for dataloader in self.dataloaders:
                for imgs, _ in dataloader:            # 不需要 label
                    output   = self.model(imgs.to(self.device))
                    mean_vec = output.mean(dim=0)     # φ：batch output 的平均向量

                    # L 個隨機單位球面向量 ξ，shape: (L, output_dim)
                    xi_vecs = sample_spherical(self.L, output.shape[-1]).T.to(self.device).float()

                    # 對每個 ξ 計算內積 ρ = ξᵀφ，backward 取梯度，累加後取平均
                    total_scalar = torch.zeros(1, device=self.device)
                    for xi in xi_vecs:
                        self.model.zero_grad()
                        rho = torch.dot(xi, mean_vec)
                        rho.backward(retain_graph=True)
                        for n, p in self.model.named_parameters():
                            if p.grad is not None:
                                precision_matrices[n].data += p.grad.abs() / (num_data * self.L)

        return precision_matrices

    def penalty(self, model):
        loss = 0
        for n, p in model.named_parameters():
            loss += (self._precision_matrices[n] * (p - self._state_params[n]) ** 2).sum()
        return loss

    def update(self, model):
        return


#### SCP — 執行訓練

In [ ]:
print('RUN SCP')
model     = Model().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)

lll_object = scp(model=model, dataloaders=[None], L=100, device=device)
lll_lambda = 100
scp_acc    = []
task_bar   = tqdm.auto.trange(args.task_number, desc='Task 1')

for i in task_bar:
    model, _, acc_list = train(
        model, optimizer, train_dataloaders[i], args.epochs_per_task,
        lll_object, lll_lambda, test_dataloaders[:i+1], evaluate, device)

    lll_object = scp(model=model, dataloaders=test_dataloaders[:i+1], L=100, device=device)
    optimizer  = torch.optim.Adam(model.parameters(), lr=args.lr)
    scp_acc.extend(acc_list)
    task_bar.set_description_str(f'Task {i+2}')

print(f'SCP 最終 Average Accuracy: {scp_acc[-1]:.2f}%')
print('=' * 80)


## 結果比較

所有方法的 Average Accuracy 隨 epoch 變化曲線（灰色虛線為 Task 切換點）。

In [ ]:
methods = {
    'Baseline': baseline_acc,
    'EWC':      ewc_acc,
    'MAS':      mas_acc,
    'SI':       si_acc,
    'RWalk':    rwalk_acc,
    'SCP':      scp_acc,
}

plt.figure(figsize=(13, 5))
x = np.arange(1, len(baseline_acc) + 1)
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

for (name, accs), color in zip(methods.items(), colors):
    plt.plot(x, accs, label=name, color=color, linewidth=2)

# 標記每個 Task 的切換點
for t in range(1, args.task_number):
    xpos = t * args.epochs_per_task
    plt.axvline(x=xpos, color='gray', linestyle='--', alpha=0.5, linewidth=1)
    plt.text(xpos + 0.3, plt.ylim()[0] + 0.5, f'Task {t+1}', fontsize=8, color='gray')

plt.xlabel('累計 Epoch 數')
plt.ylabel('Average Accuracy (%)')
plt.title('Lifelong Learning — 各方法 Average Accuracy 比較（Permuted MNIST）')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('lifelong_learning_comparison.png', dpi=150)
plt.show()

# 最終數字摘要
print('\n=== 最終 Average Accuracy（第 5 個 Task 結束後）===')
for name, accs in methods.items():
    print(f'  {name:10s}: {accs[-1]:.2f}%')
